In [1]:
import pandas as pd
from scipy.stats import linregress

def calcular_degradacao_pneus(csv_entrada, csv_saida):
    print("Calculando o desgaste por stint")

    # 1. Carrega a base estratégica
    df = pd.read_csv(csv_entrada, sep=';', decimal=',')

    # 2. Filtramos apenas as voltas válidas!
    # Ignoramos In-Laps, Out-Laps e Outliers (Safety Car, erros, etc)
    df_validas = df[(df['Tipo_Volta'] == 'Push') & (df['Outlier'] == False)].copy()

    # 3. Criamos um contador de "Volta dentro do Stint" (1, 2, 3...)
    df_validas['Volta_No_Stint'] = df_validas.groupby(['Carro', 'Stint']).cumcount() + 1

    resultados = []

    # 4. Agrupamos os dados por Carro e por Stint para analisar cada jogo de pneus
    for (carro, stint), grupo in df_validas.groupby(['Carro', 'Stint']):
        
        # Só calculamos a degradação se o piloto deu pelo menos 5 voltas limpas no pneu
        # Menos que isso, a estatística fica distorcida
        if len(grupo) >= 5:
            x = grupo['Volta_No_Stint'].values
            y = grupo['Lap Tm (Segundos)'].values

            # Regressão Linear do SciPy
            slope, intercept, r_value, p_value, std_err = linregress(x, y)

            resultados.append({
                'Carro': carro,
                'Stint': stint,
                'Degradacao (s/volta)': round(slope, 3), # Quanto tempo perde por volta
                'Ritmo_Base (s)': round(intercept, 3),   # O ritmo potencial do carro
                'Voltas_Limpas': len(grupo),             # Quantas voltas usamos na conta
                'Consistencia_R2': round(r_value**2, 3)  # O quão reloginho o piloto foi
            })

    # 5. Transforma os resultados em uma tabela e salva
    df_deg = pd.DataFrame(resultados)
    df_deg.to_csv(csv_saida, index=False, sep=';', decimal=',')
    
    print(f"Cálculo finalizado! Relatório de degradação salvo em: {csv_saida}")

# --- ÁREA DE EXECUÇÃO ---
arquivo_estrategia = '../data/03_processed/TELEMETRIA_ESTRATEGIA_T2.csv'
arquivo_relatorio_deg = '../data/03_processed/RELATORIO_DEGRADACAO_T2.csv'

calcular_degradacao_pneus(arquivo_estrategia, arquivo_relatorio_deg)

Calculando o desgaste por stint
Cálculo finalizado! Relatório de degradação salvo em: ../data/03_processed/RELATORIO_DEGRADACAO_T2.csv


In [2]:
import pandas as pd
from scipy.stats import linregress

def calcular_degradacao_pura(csv_entrada, constante_combustivel=0.015):
    print(f"Isolando Efeito Combustível (Ganho estimado: {constante_combustivel}s/volta)")
    print("-" * 75)

    df = pd.read_csv(csv_entrada, sep=';', decimal=',')
    df_validas = df[(df['Tipo_Volta'] == 'Push') & (df['Outlier'] == False)].copy()
    df_validas['Volta_No_Stint'] = df_validas.groupby(['Carro', 'Stint']).cumcount() + 1

    resultados = []

    for (carro, stint), grupo in df_validas.groupby(['Carro', 'Stint']):
        if len(grupo) >= 3:
            x = grupo['Volta_No_Stint'].values
            y_cronometro = grupo['Lap Tm (Segundos)'].values
            
            # O Tempo Corrigido (Isolando o Pneu)
            # Somamos a penalidade do peso que ele "perdeu" para simular o carro sempre pesado
            y_pneu_puro = y_cronometro + (x * constante_combustivel)

            # Regressão 1: A Degradação Aparente (O que todos veem)
            slope_aparente, _, _, _, _ = linregress(x, y_cronometro)
            
            # Regressão 2: A Degradação Pura da Borracha (O que só nós vemos)
            slope_puro, intercept_puro, r_value, _, _ = linregress(x, y_pneu_puro)

            resultados.append({
                'Carro': carro,
                'Stint': stint,
                'Deg_Aparente (s/v)': round(slope_aparente, 3), 
                'Deg_Pura_Pneu (s/v)': round(slope_puro, 3), # O Verdadeiro Desgaste
                'Ritmo_Base_Corrigido (s)': round(intercept_puro, 3),   
                'Voltas_Limpas': len(grupo)
            })

    df_deg_pura = pd.DataFrame(resultados)
    
    # Exibir resultados de forma limpa no terminal
    print(df_deg_pura.to_string(index=False))
    print("-" * 75)
    return df_deg_pura

# --- ÁREA DE EXECUÇÃO ---
arquivo_estrategia = '../data/03_processed/TELEMETRIA_ESTRATEGIA_T2.csv'

# Rodando o novo motor!
df_final = calcular_degradacao_pura(arquivo_estrategia, constante_combustivel=0.015)

Isolando Efeito Combustível (Ganho estimado: 0.015s/volta)
---------------------------------------------------------------------------
 Carro  Stint  Deg_Aparente (s/v)  Deg_Pura_Pneu (s/v)  Ritmo_Base_Corrigido (s)  Voltas_Limpas
     0      2              -0.030               -0.015                    82.785              3
     0      3              -0.519               -0.504                    83.149              3
     1      1              -0.347               -0.332                    83.139              3
     4      1              -2.096               -2.081                    88.132              3
     6      2              -0.027               -0.012                    83.084              4
     7      1               0.032                0.047                    82.834              4
     7      3              -3.847               -3.832                    94.555              3
     8      1              -2.589               -2.574                    89.487              3
 